<img style="float: center;" src='https://github.com/spacetelescope/jwst-pipeline-notebooks/raw/main/_static/stsci_header.png' alt="stsci_logo" width="900px"/>

# Repackaging RWDDT Checkpoint Outputs

**Authors**: Taylor James Bell (ESA/AURA for STScI)<br>
**Last Updated**: May 27, 2026<br>
**Eureka! Pipeline Version**: https://github.com/kevin218/Eureka/tree/tjb_rwddt

**Purpose**:<br/>

This checkpoint notebook is intentionally a thin configuration layer. Put the
per-eclipse Stage-2/3/4 files in `visits`, then provide the shared Stage-5
samples, shared Stage-5 fit table, EPF, and ECF.

The helper module handles shared versus channel-specific samples using Eureka's
implicit `ch0` convention:

- visit 1 -> implicit ch0 -> unsuffixed parameter names
- visit 2 -> `_ch1`
- visit 3 -> `_ch2`
- etc.

Timing priority inside `eclipse_cat_builder.py` is:

1. Use direct `t_secondary`, `t_secondary_ch#`, or fixed EPF values if present.
2. Otherwise derive `t_secondary` from `t0`, `per`, `ecosw`, `esinw`, `b`,
   `a`, and `Rs`, using samples when available and EPF fixed values as
   fallbacks.
3. Apply the ECF `compute_ltt` setting if present; otherwise default to the
   secondary-eclipse behavior (`True`).

In [ ]:
from pathlib import Path
import xarray as xr
xr.set_options(
    display_max_html_elements=2000,
    display_max_children=30,
    display_max_items=100,
)

import eclipse_cat_builder as ecb

## Configure the checkpoint

In [ ]:
checkpoint = 2

stage5_samples = Path('YourPathHere_shared_samples.h5')
stage5_fit = Path('YourPathHere_shared_Table_Save_ch0.txt')

# Optional but recommended for fits where timing is controlled by t0/per,
# ecosw/esinw, or fixed values in the EPF. Set to None if unavailable.
stage5_epf = Path('YourPathHere.epf')
stage5_ecf = Path('YourPathHere.ecf')

STAR = 'YourStarHere'       # E.g., 'GJ 3929'
PLANET = 'YourPlanetHere'   # E.g., 'GJ 3929 b'
HLSPVER = '0.1'  # Increment to '1.0' once approved for publication.
outputDirectory = Path('./saves/')

## List the per-eclipse inputs

In [ ]:
visits = [
    {
        'visit': 1,
        'stage2_fits': Path('Eclipse1_calints.fits'),
        'stage3_specdata': Path('Eclipse1_SpecData.h5'),
        'stage4_lcdata': Path('Eclipse1_LCData.h5'),
        'stage4cal': Path('Eclipse1_CalStellarSpec.h5'),
        'SRC_DOI': 'Get This From Taylor',
    },
    {
        'visit': 2,
        'stage2_fits': Path('Eclipse2_calints.fits'),
        'stage3_specdata': Path('Eclipse2_SpecData.h5'),
        'stage4_lcdata': Path('Eclipse2_LCData.h5'),
        'stage4cal': Path('Eclipse2_CalStellarSpec.h5'),
        'SRC_DOI': 'Get This From Taylor',
    },
]

# Add more visit dictionaries as needed.

## Build the checkpoint products

In [ ]:
outputs = ecb.build_checkpoint_products(
    visits=visits,
    stage5_samples=stage5_samples,
    stage5_fit=stage5_fit,
    stage5_epf=stage5_epf,
    stage5_ecf=stage5_ecf,
    checkpoint=checkpoint,
    STAR=STAR,
    PLANET=PLANET,
    HLSPVER=HLSPVER,
    out_dir=outputDirectory,
    make_checkpoint_plots=True,
)

{k: v for k, v in outputs.items() if isinstance(v, (str, type(None)))}

## Inspect the high-level checkpoint catalog product

In [ ]:
outputs['catalog_dataset']

## Inspect the per-visit intermediate light-curve products

In [ ]:
outputs['lightcurve_datasets']

## Make one pair of report figures per eclipse just for sanity checking

In [ ]:
figure_paths = {}

lc_tree = outputs["lightcurve_datasets"]

for node_name, node in lc_tree.children.items():
    # Skip any non-visit child nodes, if present.
    if not node_name.startswith("visit_"):
        continue

    lc_ds = node.ds
    visit = int(lc_ds.visit.values.item())

    figures = ecb.make_lightcurve_figures(
        lc_ds,
        planet=PLANET,
        out_dir=outputDirectory,
        prefix=f"ecl{visit:03d}_",
    )

    figure_paths[visit] = {
        key: value for key, value in figures.items()
        if isinstance(value, str)
    }

figure_paths